In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.animation
from palettable.cartocolors.qualitative import Antique_10, Prism_10
from scipy.signal import savgol_filter
from tqdm import tqdm
import matplotlib.patheffects as pe

In [ ]:
def extract_and_order_snapshotIdx(rawtree, branch):
    #this function extract only the snapshot key (i.e. the integer value) from the rawtree halotree output
    keys = list(rawtree[branch].keys())
    snapshotIdx = [x for x in keys if not isinstance(x, str)]
    snapshotIdx.sort()
    return snapshotIdx

In [ ]:
def get_r_oden(rawtree, branch, idx, oden):
    halo = rawtree[branch][idx]
    r_odenkey = 'r%s' % oden
    #this function return r200 value or find the closest value to it (in case the halo does not have 'r200' radius)
    if r_odenkey in halo.keys():
        r_oden = halo[r_odenkey]
    else:
        key_list = list(halo.keys())
        r_keys = np.array([x[1:] for x in key_list if x[0] =='r'])
        r_key = r_keys[abs(r_keys.astype(float)-oden)==abs(r_keys.astype(float)-oden).min()][0]
        r_oden = halo['r'+r_key]
    return r_oden

def get_r200(rawtree, branch, idx, halo_radius = False):
    if halo_radius == False:
        if (get_r_oden(rawtree, branch, idx, 200) < get_r_oden(rawtree, branch, idx, 250)):
            #print('r200 error at Idx %s and Branch %s' % (idx, branch))
            if ((rawtree[branch][idx]['Halo_Radius'] <= get_r_oden(rawtree, branch, idx, 150)) and (rawtree[branch][idx]['Halo_Radius'] >= get_r_oden(rawtree, branch, idx, 250))) and rawtree[branch][idx]['cden'] < 250 and rawtree[branch][idx]['cden'] > 150:
                #print('Replaced by Halo_Radius')
                return rawtree[branch][idx]['Halo_Radius']
            elif get_r_oden(rawtree, branch, idx, 150) >= get_r_oden(rawtree, branch, idx, 250):
                #print('Replaced by closest value to r150')
                return get_r_oden(rawtree, branch, idx, 150)
            elif get_r_oden(rawtree, branch, idx, 150) < get_r_oden(rawtree, branch, idx, 250):
                #print('Replaced by closest value to r250')
                return get_r_oden(rawtree, branch, idx, 250)
        else:
            return get_r_oden(rawtree, branch, idx, 200)
    else:
        return rawtree[branch][idx]['Halo_Radius']

In [ ]:
def pos_from_IDs(ID, metadata):
    ID_all = metadata['ID']
    pos_all = metadata['pos']
    return pos_all[np.intersect1d(ID, ID_all, return_indices=True)[2]]

In [ ]:
codetp = 'GADGET4'
halotree_ver = 2013

halo_dir = '/work/hdd/bezm/gtg115x/Halo_Finding/%s' % codetp
metadata_dir = '/work/hdd/bezm/tnguyen2/AGORA/%s/metadata' % codetp
rawtree = np.load(halo_dir + '/halotree_%s_final.npy' % halotree_ver, allow_pickle=True).tolist()
pfs = np.loadtxt(halo_dir + '/pfs_allsnaps_%s.txt' % halotree_ver, dtype=str)[:,0]
zlist = np.loadtxt(halo_dir + '/pfs_allsnaps_%s.txt' % halotree_ver, dtype=str)[:,1]
tlist = np.loadtxt(halo_dir + '/pfs_allsnaps_%s.txt' % halotree_ver, dtype=str)[:,2].astype(float)

assignment = np.load(halo_dir + '/' + 'star_id_%s_final.npy' % halotree_ver, allow_pickle=True).tolist()
assignment = assignment['ids']

#### Run the following codes to visualize the star assignment results and the DM halos surrounding the "focus_branch" (in this case Halo '0') --> This helps identify the secondary halo of the major merger. The halos with dashed-line boundary are ones without stars bound to them.

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15,5))

matplotcolors = list(matplotlib.colors.TABLEAU_COLORS)[0:3] + list(matplotlib.colors.TABLEAU_COLORS)[4:]
colors = matplotcolors + Antique_10.mpl_colors + Prism_10.mpl_colors
color_dict = {}
focus_branch = '0'
alpha = 0.5 #set the transparency of the star particles
zoom_factor = 2 #set the zoom factor for the plot, relative to the R200 radius
idx = 110 #the snapshot index
halo_radius_switch = False

for proj in [0, 1, 2]:
    axs[proj].clear()
    if proj == 0:
        axis1, axis2 = 0, 1
    elif proj == 1:
        axis1, axis2 = 1, 2
    elif proj == 2:
        axis1, axis2 = 2, 0
    #
    xlim = [rawtree[focus_branch][idx]['Halo_Center'][axis1] - zoom_factor*get_r200(rawtree, focus_branch, idx, halo_radius_switch), rawtree[focus_branch][idx]['Halo_Center'][axis1] + zoom_factor*get_r200(rawtree, focus_branch, idx, halo_radius_switch)]
    ylim = [rawtree[focus_branch][idx]['Halo_Center'][axis2] - zoom_factor*get_r200(rawtree, focus_branch, idx, halo_radius_switch), rawtree[focus_branch][idx]['Halo_Center'][axis2] + zoom_factor*get_r200(rawtree, focus_branch, idx, halo_radius_switch)]
    #
    allstars = np.load(metadata_dir + '/' + 'star_metadata_allbox_%s.npy' % idx, allow_pickle=True).tolist()
    allpos = allstars['pos']
    allID = allstars['ID'].astype(int)
    boundID = np.array([])
    #
    for branch in assignment.keys():
        if idx not in rawtree[branch].keys():
            continue
        if rawtree[branch][idx]['Halo_Mass']/rawtree[focus_branch][idx]['Halo_Mass'] < 0.1:
            continue
        if branch not in color_dict.keys():
            color_dict[branch] = np.random.randint(0, len(colors))
        if idx not in assignment[branch].keys():
            if branch == focus_branch:
                circle = plt.Circle((rawtree[branch][idx]['Halo_Center'][axis1], rawtree[branch][idx]['Halo_Center'][axis2]), get_r200(rawtree, branch, idx, halo_radius_switch), color='black', fill=False, label=branch, linestyle='--')
                axs[proj].add_patch(circle)
                axs[proj].text(x=rawtree[branch][idx]['Halo_Center'][axis1], y=rawtree[branch][idx]['Halo_Center'][axis2], s=branch, c='black', path_effects=[pe.withStroke(linewidth=2, foreground="yellow")])
            else:
                circle = plt.Circle((rawtree[branch][idx]['Halo_Center'][axis1], rawtree[branch][idx]['Halo_Center'][axis2]), get_r200(rawtree, branch, idx, halo_radius_switch), color=colors[color_dict[branch]], fill=False, label=branch, linestyle='--')
                axs[proj].add_patch(circle)
                if rawtree[branch][idx]['Halo_Center'][axis1] > xlim[0] and rawtree[branch][idx]['Halo_Center'][axis1] < xlim[1] and rawtree[branch][idx]['Halo_Center'][axis2] > ylim[0] and rawtree[branch][idx]['Halo_Center'][axis2] < ylim[1]:
                    axs[proj].text(x=rawtree[branch][idx]['Halo_Center'][axis1], y=rawtree[branch][idx]['Halo_Center'][axis2], s=branch, c=colors[color_dict[branch]], path_effects=[pe.withStroke(linewidth=2, foreground="black")])
            continue
        #if np.linalg.norm(rawtree_snapFirst[idx][branch]['Halo_Center'] - rawtree[focus_branch][idx]['Halo_Center']) > zoom_factor*rawtree[focus_branch][idx]['Halo_Radius']:
        #    continue
        if branch == focus_branch:
            circle = plt.Circle((rawtree[branch][idx]['Halo_Center'][axis1], rawtree[branch][idx]['Halo_Center'][axis2]), get_r200(rawtree, branch, idx, halo_radius_switch), color='black', fill=False, label=branch)
            axs[proj].add_patch(circle)
            if len(assignment[branch][idx]) > 0:
                axs[proj].scatter(pos_from_IDs(assignment[branch][idx], allstars)[:,axis1], pos_from_IDs(assignment[branch][idx], allstars)[:,axis2], marker='*', c='black',s=1, alpha=alpha)
            boundID = np.append(boundID, assignment[branch][idx])
        else:
            circle = plt.Circle((rawtree[branch][idx]['Halo_Center'][axis1], rawtree[branch][idx]['Halo_Center'][axis2]), get_r200(rawtree, branch, idx, halo_radius_switch), color=colors[color_dict[branch]], fill=False, label=branch)
            axs[proj].add_patch(circle)
            if rawtree[branch][idx]['Halo_Center'][axis1] > xlim[0] and rawtree[branch][idx]['Halo_Center'][axis1] < xlim[1] and rawtree[branch][idx]['Halo_Center'][axis2] > ylim[0] and rawtree[branch][idx]['Halo_Center'][axis2] < ylim[1]:
                axs[proj].text(x=rawtree[branch][idx]['Halo_Center'][axis1], y=rawtree[branch][idx]['Halo_Center'][axis2], s=branch, c=colors[color_dict[branch]], fontweight='bold', path_effects=[pe.withStroke(linewidth=2, foreground="black")])
            if len(assignment[branch][idx]) > 0:
                axs[proj].scatter(pos_from_IDs(assignment[branch][idx], allstars)[:,axis1], pos_from_IDs(assignment[branch][idx], allstars)[:,axis2], marker='*', c=colors[color_dict[branch]], s=1, alpha=alpha)
            boundID = np.append(boundID, assignment[branch][idx])
    lossID = np.setdiff1d(allID, boundID)
    losspos = allpos[np.intersect1d(lossID, allID, return_indices=True)[2]]
    axs[proj].scatter(losspos[:,axis1], losspos[:,axis2], marker='X', c='red', s=1)
    if proj == 0:
        axs[proj].set_xlabel('x')
        axs[proj].set_ylabel('y')
    elif proj == 1:
        axs[proj].set_xlabel('y')
        axs[proj].set_ylabel('z')
        axs[proj].set_title('Snapshot %s, z = %.2f' % (idx,float(zlist[idx])))
    elif proj == 2:
        axs[proj].set_xlabel('z')
        axs[proj].set_ylabel('x')
    axs[proj].set_aspect('equal', adjustable='box')
    axs[proj].set(xlim=xlim, ylim=ylim)

plt.tight_layout()